In [3]:
import pandas as pd

In [4]:
# Loading the list customers

def load_customers(data_dir: str) -> pd.DataFrame:
    customers = pd.read_csv(f"{data_dir}/olist_customers_dataset.csv")
    orders = pd.read_csv(f"{data_dir}/olist_orders_dataset.csv")

    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

    merged = customers.merge(orders, on = "customer_id")

    result = merged.groupby("customer_unique_id").agg(
        first_order_date = ("order_purchase_timestamp", "min"),
        last_order_date = ("order_purchase_timestamp", "max"),
    ).reset_index()

    return result

In [5]:
# Checking the dataset

unique_customers_aggregated = load_customers("../data/raw")  # adjust path to your CSVs
print(unique_customers_aggregated.shape)
unique_customers_aggregated.head()

(96096, 3)


,customer_unique_id,first_order_date,last_order_date
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42


In [15]:
# Generating sessions (# of sessions + device_type per user)

import numpy as np
import uuid
from datetime import timedelta

def sessions_df (customers_df):
    rows = []
    for _, customer in customers_df.iterrows():
        num_sessions = np.random.randint(1, 6)
        for _ in range(num_sessions):
            rows.append({
                "customer_unique_id": customer["customer_unique_id"],
                "session_id": str(uuid.uuid4()),
                "session_start": customer["first_order_date"] - timedelta(seconds = np.random.randint(0, 30 * 24 * 60 * 60)),
                "device_type": np.random.choice(["mobile", "desktop", "tablet"])
            })
    sessions_df = pd.DataFrame(rows)
    return sessions_df

In [17]:
sessions_df  = sessions_df (unique_customers_aggregated)
print(sessions_df .shape)
sessions_df.head()

(288443, 4)


,customer_unique_id,session_id,session_start,device_type
0,0000366f3b9a7992bf8c76cfdf3221e2,c6adfd02-f743-4875-b510-edbbfc9f1af2,2018-04-11 01:42:33,mobile
1,0000b849f77a49e4a4ce2b2a4ca5be3f,553ae655-f93a-402d-9841-39e2467f3cf0,2018-04-24 06:44:22,desktop
2,0000b849f77a49e4a4ce2b2a4ca5be3f,9d6fcdb1-126c-460b-882d-2535e8b02257,2018-04-26 08:06:00,tablet
3,0000b849f77a49e4a4ce2b2a4ca5be3f,699bdcfc-0654-41ca-b52d-6b67a0c7ed62,2018-05-06 06:51:02,mobile
4,0000b849f77a49e4a4ce2b2a4ca5be3f,a0390a8f-4b4c-4054-ac00-1fdbb47bb6f3,2018-04-16 03:42:32,tablet


In [26]:
# Generate funnel events - steps a user moves through on an e-commerce site
def generate_funnel_events(sessions_df):
    rows = []
    funnel_steps = ["page_view", "product_view", "add_to_cart", "checkout", "purchase"]
    drop_off_prob = 0.3  # 30% chance of stopping at each step
    for index, session in sessions_df.iterrows():
        events = []
        current_time = session["session_start"]
        for step in funnel_steps:
            current_time = current_time + timedelta(minutes = np.random.randint(1,10))
            events.append((step, current_time))
            if np.random.random() < drop_off_prob:
                break
        for step, timestamp in events:
            rows.append({
                "session_id": session["session_id"],
                "customer_unique_id": session["customer_unique_id"],
                "event_type": step,
                "event_timestamp": timestamp
            })
    events_df = pd.DataFrame(rows)
    return events_df


In [27]:
events_df = generate_funnel_events(sessions_df)
print(events_df.shape)
events_df.head(10)

(798928, 4)


,session_id,customer_unique_id,event_type,event_timestamp
0,c6adfd02-f743-4875-b510-edbbfc9f1af2,0000366f3b9a7992bf8c76cfdf3221e2,page_view,2018-04-11 01:49:33
1,c6adfd02-f743-4875-b510-edbbfc9f1af2,0000366f3b9a7992bf8c76cfdf3221e2,product_view,2018-04-11 01:53:33
2,c6adfd02-f743-4875-b510-edbbfc9f1af2,0000366f3b9a7992bf8c76cfdf3221e2,add_to_cart,2018-04-11 01:55:33
3,c6adfd02-f743-4875-b510-edbbfc9f1af2,0000366f3b9a7992bf8c76cfdf3221e2,checkout,2018-04-11 02:04:33
4,c6adfd02-f743-4875-b510-edbbfc9f1af2,0000366f3b9a7992bf8c76cfdf3221e2,purchase,2018-04-11 02:09:33
5,553ae655-f93a-402d-9841-39e2467f3cf0,0000b849f77a49e4a4ce2b2a4ca5be3f,page_view,2018-04-24 06:53:22
6,9d6fcdb1-126c-460b-882d-2535e8b02257,0000b849f77a49e4a4ce2b2a4ca5be3f,page_view,2018-04-26 08:11:00
7,699bdcfc-0654-41ca-b52d-6b67a0c7ed62,0000b849f77a49e4a4ce2b2a4ca5be3f,page_view,2018-05-06 06:59:02
8,699bdcfc-0654-41ca-b52d-6b67a0c7ed62,0000b849f77a49e4a4ce2b2a4ca5be3f,product_view,2018-05-06 07:00:02
9,699bdcfc-0654-41ca-b52d-6b67a0c7ed62,0000b849f77a49e4a4ce2b2a4ca5be3f,add_to_cart,2018-05-06 07:03:02
